In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pwlf
import pandas as pd
import statsmodels.api as sm

In [2]:
def testeAceleracao(prefixo, xNew,yNew):
	#x = data.Odometer.values
	#x = data.Seconds.values
	#y = data.Velocity.values

	#x= x[10:-10]
	#y= y[10:-10]

	#interpolar dados para deixar mesmo número de pontos por distância
	#xNew = np.linspace(x[0],x[-1],200)
	#yNew = np.interp(xNew,x,y)

	acc = np.diff(yNew,prepend=yNew[0])
 
	acc = acc / (xNew[1]-xNew[0])
 
	accSmooth = sm.nonparametric.lowess(acc,xNew, is_sorted=True, return_sorted=False, frac=0.2)

	limiarACC = 0.005
	limiarACC = np.max(acc)*0.01

	for posZC in range(0, len(accSmooth)):
		#encontrar valor que cruza o zero
		if accSmooth[posZC] < limiarACC:
			break	
 
	# PLOTAR GRAFICO DE ACELERACAO
 
	plt.figure(figsize=(5,5))
	plt.plot(xNew,acc,'r')
	plt.plot(xNew,accSmooth,'b')
	plt.legend(['raw', 'filter'])
	plt.plot(xNew[posZC], accSmooth[posZC],'og')
	plt.title('Aceleração')
	plt.savefig('./figuras/' + prefixo + '_acc.png')
	plt.close()

	return posZC
 

In [3]:
def findIndexOld(xNew, xOld, indexNew):
    #percorrer de 0 a final o xOld e encontrar ponto mais próximo do xNew
    flgFind = False
    for index,ponto in enumerate(xOld):
        if (index > 0):
            if ponto >= xNew[indexNew]:
                flgFind = True
                d1 = np.abs(xNew[indexNew] - xOld[index])
                d2 = np.abs(xNew[indexNew] - xOld[index-1])
                if d1 <= d2:
                    indexOut = index
                else:
                    indexOut = index -1
                break
    return indexOut

In [4]:
def rotinaEncontrarCurvas(identificador, data, maxCurvas, limiarParada):
    print('PROCESSANDO ---- ' + identificador)
    x = data.Odometer.values
    #x = data.Seconds.values
    y = data.Velocity.values
    tempo = data.Seconds.values

    #x= x[10:-10]
    #y= y[10:-10]
    #tempo = tempo[10:-10]
    
    tempoTotal = tempo[-1]

    #interpolar dados para deixar mesmo número de pontos por distância
    xNew = np.linspace(x[0],x[-1],200)
    yNew = np.interp(xNew,x,y)
    
    velocidadeMedia = np.mean(yNew)

    posZC = testeAceleracao(identificador, xNew, yNew)
    #posZC = testeVelocidade(data)

    # PLOT VALORES INTERPOLADOS

    plt.figure(figsize=(5,5))
    plt.plot(x,y,'b')
    plt.plot(xNew,yNew,'r')
    plt.plot(xNew[posZC],yNew[posZC],'og')
    plt.legend(['original', 'interpolado'])
    plt.savefig('./figuras/' + identificador + '_interpolacao.png')
    plt.close()

    xOld = xNew
    yOld = yNew

    xNew = xNew[posZC:]
    yNew = yNew[posZC:]

    listaCurvas = range(1,maxCurvas+1)
    dadosCurvas = []
    diffPercentSSR = 0
    for indexCurvas, nCurvas in enumerate(listaCurvas):
        print(nCurvas)
        outCurvas = []
        dictCurvas = {'dict': 0,
                     'slopes' : 1,
                    'breaks' : 2,
                    'r2' : 3,
                    'ssr': 4
                    }

        outCurvas.append(dictCurvas)
        # initialize piecewise linear fit with your x and y data
        my_pwlf = pwlf.PiecewiseLinFit(xNew, yNew)
        # fit the data for four line segments
        res = my_pwlf.fit(nCurvas)

        # predict for the determined points
        xHat = np.linspace(min(xNew), max(xNew), num=200)
        yHat = my_pwlf.predict(xHat)

        outCurvas.append(my_pwlf.slopes)
        outCurvas.append(my_pwlf.fit_breaks)
        outCurvas.append(my_pwlf.r_squared())
        outCurvas.append(my_pwlf.ssr)

        if indexCurvas ==0:
            # plot the results
            plt.figure(figsize=(10,5))
            plt.title(identificador)
            plt.plot(xOld[:posZC], yOld[:posZC],'r')
            plt.plot(xNew, yNew, 'o')
            plt.plot(xHat, yHat, '-')
            plt.savefig('./figuras/' + identificador + '_curva_' + str(indexCurvas +1) + '.png')
            plt.close()
            dadosCurvas.append(outCurvas)

        else:
            #calcular diferença percentual
            dadoPrev = dadosCurvas[-1]
            dadoNovo = outCurvas
            diffPercentSSR = (dadoNovo[dictCurvas['ssr']] - dadoPrev[dictCurvas['ssr']])/dadoPrev[dictCurvas['ssr']]*100
            diffPercentR2 = (dadoNovo[dictCurvas['r2']] - dadoPrev[dictCurvas['r2']])/dadoPrev[dictCurvas['r2']]*100
            print('SSR --- INDEX {:d} X {:d} == {:.2f}'.format(indexCurvas+1, indexCurvas-1+1, diffPercentSSR))
            print('R2  --- INDEX {:d} X {:d} == {:.2f}'.format(indexCurvas+1, indexCurvas-1+1, diffPercentR2))
            if np.abs(diffPercentSSR) < limiarParada:
                print('SAINDO LOOP ....')
                break
            else:
                print("MELHORA NA REGRESSAO")
                #adicionar dados a fila
                dadosCurvas.append(outCurvas)
                
                # PLOTAR CURVA COM RETAS
                #https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html
                
                plt.figure(figsize=(10,5))
                plt.title('')
                plt.plot(xOld[:posZC], yOld[:posZC],'r')
                plt.plot(xNew, yNew, 'bo', markersize=3)
                plt.plot(xHat, yHat, '-', color='black', linewidth=2)
                plt.grid(axis='x', color='0.95')
                plt.legend(['Acceleration Phase','','Phases'])
                plt.xlabel('Distance (m)')
                plt.ylabel('Velocity (m/s)')
                
                plt.savefig('./figuras/' + identificador + '_curva_' + str(indexCurvas+1) + '.png', dpi=1200)
                #plt.savefig('./figuras/' + identificador + '_curva_' + str(indexCurvas+1) + '.png')
                plt.close()

    listaSSR = []
    listaR2 = []
    for dado in dadosCurvas:
        dicionario = dado[0]
        listaSSR.append(dado[dicionario['ssr']])
        listaR2.append(dado[dicionario['r2']])


    velR2 = np.diff(listaR2)/listaR2[:-1] * 100
    velSSR = np.diff(listaSSR)/listaSSR[:-1] * 100

    #ajustando para novo vetor de curvas
    listaCurvas = np.linspace(1, len(dadosCurvas), len(dadosCurvas))

    # PLOTAR CURVA DE VALORES R2
    
    plt.figure(figsize=(10,10))
    plt.subplot(2,1,1)
    plt.plot(listaCurvas, listaR2, '.r')
    plt.title('Variação R2')
    plt.xlabel('Número de retas')
    plt.subplot(2,1,2)
    plt.plot(listaCurvas[1:], velR2, '.b')
    plt.title('Variação percentual R2')
    plt.xlabel('Número de retas')
    plt.savefig('./figuras/' + identificador + '_R2.png')
    plt.close()
    
    # PLOTAR CURVA DE VALORES SSR

    plt.figure(figsize=(10,10))
    plt.subplot(2,1,1)
    plt.plot(listaCurvas, listaSSR,'.r')
    plt.title('Variação SSR --- ' + identificador)
    plt.xlabel('Número de retas')
    plt.subplot(2,1,2)
    plt.plot(listaCurvas[1:], velSSR, '.b')
    plt.title('Variação percentual SSR --- ' + identificador)
    plt.xlabel('Número de retas')
    plt.savefig('./figuras/' + identificador + '_SSR.png')
    plt.close()

    melhorCurva = dadosCurvas[-1]
    
    dictOutput = {'prefixo': 0,
                  'tempoTotal': 1,
                  'velMedia': 2,
                  'nZonas': 3,
                  'dadosCurvas': 4,
                  'vetores': 5
                }
    
    out = []
    
    out.append(identificador)
    out.append(tempoTotal)
    out.append(velocidadeMedia)
    out.append(len(dadosCurvas))
    out.append(melhorCurva)
    out.append([tempo, x, y, xNew, yNew])

    return (dictOutput, out)

In [5]:
dados1 = pd.read_excel("ATHLETE.xlsx")
                       
listaDados = [dados1]
#listaDados = [dados1]

listaID = ['Athlete1']
#listaID = ['Aline']


FileNotFoundError: [Errno 2] No such file or directory: 'ATHLETE.xlsx'

In [ ]:
limitepercent = 40
maxCurvas = 5
listaPrefixos = []
listaTempoProva = []
listaVelMedia = []
listaNZonas = []
listaSSR = []
listaR2 = []
listaResultados = []
listaVetores = []

for index, data in enumerate(listaDados):
    (dictOutput, out) = rotinaEncontrarCurvas(listaID[index], data, maxCurvas, limitepercent)
    listaPrefixos.append(out[dictOutput['prefixo']])
    listaTempoProva.append(out[dictOutput['tempoTotal']])
    listaVelMedia.append(out[dictOutput['velMedia']])
    listaNZonas.append(out[dictOutput['nZonas']])
    resultado = out[dictOutput['dadosCurvas']]
    vetores = out[dictOutput['vetores']]
    
    dictResultado = resultado[0]
    listaSSR.append(resultado[dictResultado['ssr']])
    listaR2.append(resultado[dictResultado['r2']])
    listaResultados.append(resultado)
    listaVetores.append(vetores)
    
#criar a lista de colunas
numProcessados = len(listaPrefixos)
listaAnalisesZona = ['PontoInicio', 'PontoFinal','Inclinacao', 'TempoInicio', 'TempoFinal', 'VelMed','VelInicio','VelFinal']
numAnalisesZona = len(listaAnalisesZona)
matrizResultados = np.zeros((numProcessados, maxCurvas, numAnalisesZona))

#preencher dados
for i in range(0, numProcessados): # processados
    result = listaResultados[i]
    meuVetor = listaVetores[i]
    meuTempo = meuVetor[0]
    meuXold = meuVetor[1]
    meuYold = meuVetor[2]
    meuXnew = meuVetor[3]
    meuYnew = meuVetor[4]
    
    dictResultado = result[0]
    nResult = len(result[dictResultado['slopes']])
    for j in range(0, maxCurvas):
        if j < nResult:
            #tenho dados
            valoresK = []
            break0 = result[dictResultado['breaks']][j]
            break1 = result[dictResultado['breaks']][j+1]
            
            #encontrar os indices novos
            indexBreak0 = np.argwhere(meuXnew >= break0)[0][0]
            indexBreak1 = np.argwhere(meuXnew >= break1)[0][0]
            
            indexOld0 = findIndexOld(meuXnew, meuXold, indexBreak0)
            indexOld1 = findIndexOld(meuXnew, meuXold, indexBreak1)
            
            valoresK.append(break0)
            valoresK.append(break1)
            valoresK.append(result[dictResultado['slopes']][j])
            
            valoresK.append(meuTempo[indexOld0])
            valoresK.append(meuTempo[indexOld1])
            
            valoresK.append(np.mean(meuYnew[indexBreak0: indexBreak1]))
            
            valoresK.append(meuYnew[indexBreak0])
            valoresK.append(meuYnew[indexBreak1])
            
            for k in range(0, numAnalisesZona):
                matrizResultados[i, j, k] = valoresK[k]

dadosOut = pd.DataFrame()
dadosOut['Atleta'] = listaPrefixos
dadosOut['Tempo Prova'] = listaTempoProva
dadosOut['Velocidade Média'] = listaVelMedia
dadosOut['N° Zonas'] = listaNZonas
dadosOut['SSR total'] = listaSSR
dadosOut['R2 total'] = listaR2

for j in range(0, maxCurvas):
    for k in range(0,numAnalisesZona):
        dadosOut['Zona_' + str(j+1) + '_' + listaAnalisesZona[k]] = matrizResultados[:,j,k]

display(dadosOut)
dadosOut.to_excel('analiseDadoskayak.xlsx')
#display(matrizResultados)